In [ ]:
# --- CHỈ DÙNG 1 GPU ---
# Phải đặt TRƯỚC khi import torch. Trên Kaggle "GPU T4 x2" vẫn chỉ dùng GPU số 0;
# chọn "GPU P100" (1 GPU) cũng chạy được, không cần sửa gì thêm.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "PraNet"
CHUAN_HOA = "imagenet"   # cách chuẩn hóa ảnh, giữ như notebook cũ
# ============================================================
# CELL 0: CẤU HÌNH LƯỢT CHẠY (CHỈ SỬA SEED)
# ============================================================
SEED = 0            # chạy seed 0..19, trùng seed của YOLO
IMG_SIZE = 640      # giống YOLO (imgsz=640)
EPOCHS = 100
CLOSE_MOSAIC = 10   # tắt mosaic ở 10 epoch cuối, giống YOLO (close_mosaic=10)

# Tăng cường lấy đúng giá trị trong args.yaml của 40 lượt YOLO
AUG = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0)

DATASET_ROOT = "/kaggle/input/datasets/okeydokey0805/kvasir-yolo-seg-bg20/Kvasir_YOLO_SEG_BG20"
print(f"{MODEL_NAME}: SEED={SEED}, IMG_SIZE={IMG_SIZE}, EPOCHS={EPOCHS}")

import torch
assert torch.cuda.is_available(), "❌ Chưa bật GPU: Settings > Accelerator > GPU"
print(f"GPU dùng: {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================
# CELL 1: DATASET BG20 + TĂNG CƯỜNG GIỐNG YOLO (MOSAIC, CO GIÃN, DỊCH, HSV, LẬT NGANG)
# ============================================================
# Ghi chú so với Ultralytics:
#  - Ảnh được kéo về IMG_SIZE x IMG_SIZE (không letterbox), giống cách đánh giá theo điểm ảnh của đề tài.
#  - Mosaic 4 ảnh, sau đó biến đổi affine (co giãn ±50%, dịch ±10%) cắt về IMG_SIZE, giống random_perspective
#    với degrees = shear = perspective = 0 như cấu hình YOLO.
#  - Không dùng erasing: Ultralytics chỉ dùng erasing cho bài phân loại.
import os, glob, random, cv2, torch
import numpy as np
from torch.utils.data import Dataset


def doc_anh_mat_na(img_path, label_dir):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    stem = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(label_dir, stem + ".txt")
    if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 5:  # đa giác YOLO-seg
                    coords = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
                    coords[:, 0] *= w
                    coords[:, 1] *= h
                    cv2.fillPoly(mask, [coords.astype(np.int32)], 255)
    return img, mask


def hsv_ngau_nhien(img, hgain, sgain, vgain):
    """Giống RandomHSV của Ultralytics."""
    r = np.random.uniform(-1, 1, 3) * [hgain, sgain, vgain] + 1
    hue, sat, val = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2HSV))
    x = np.arange(0, 256, dtype=r.dtype)
    lut_hue = ((x * r[0]) % 180).astype(np.uint8)
    lut_sat = np.clip(x * r[1], 0, 255).astype(np.uint8)
    lut_val = np.clip(x * r[2], 0, 255).astype(np.uint8)
    hsv = cv2.merge((cv2.LUT(hue, lut_hue), cv2.LUT(sat, lut_sat), cv2.LUT(val, lut_val)))
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def affine_ngau_nhien(img, mask, out, scale, translate):
    """Co giãn và dịch quanh tâm ảnh, cắt về out x out (random_perspective với góc xoay = 0)."""
    h, w = img.shape[:2]
    C = np.eye(3); C[0, 2], C[1, 2] = -w / 2, -h / 2
    s = random.uniform(1 - scale, 1 + scale)
    S = np.eye(3); S[0, 0] = S[1, 1] = s
    T = np.eye(3)
    T[0, 2] = random.uniform(0.5 - translate, 0.5 + translate) * out
    T[1, 2] = random.uniform(0.5 - translate, 0.5 + translate) * out
    M = (T @ S @ C)[:2]
    img = cv2.warpAffine(img, M, (out, out), flags=cv2.INTER_LINEAR, borderValue=(114, 114, 114))
    mask = cv2.warpAffine(mask, M, (out, out), flags=cv2.INTER_NEAREST, borderValue=0)
    return img, mask


class PolypBG20(Dataset):
    def __init__(self, root_dir, split, img_size, augment, chuan_hoa):
        self.img_dir = os.path.join(root_dir, "images", split)
        self.label_dir = os.path.join(root_dir, "labels", split)
        self.img_files = sorted(glob.glob(os.path.join(self.img_dir, "*.jpg")))
        assert len(self.img_files) > 0, f"Không tìm thấy ảnh tại: {self.img_dir}"
        self.s, self.augment, self.chuan_hoa = img_size, augment, chuan_hoa
        self.mosaic = augment and AUG["mosaic"] > 0   # tắt ở CLOSE_MOSAIC epoch cuối

    def __len__(self):
        return len(self.img_files)

    def anh_vuong(self, i):
        img, mask = doc_anh_mat_na(self.img_files[i], self.label_dir)
        img = cv2.resize(img, (self.s, self.s), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.s, self.s), interpolation=cv2.INTER_NEAREST)
        return img, mask

    def mosaic4(self, i):
        """Mosaic 4 ảnh giống Ultralytics: tâm ngẫu nhiên trên khung 2s x 2s rồi affine cắt về s x s."""
        s = self.s
        ids = [i] + random.choices(range(len(self)), k=3)
        yc, xc = (int(random.uniform(s // 2, 3 * s // 2)) for _ in range(2))
        khung = np.full((2 * s, 2 * s, 3), 114, dtype=np.uint8)
        kmask = np.zeros((2 * s, 2 * s), dtype=np.uint8)
        for k, j in enumerate(ids):
            img, mask = self.anh_vuong(j)
            h, w = img.shape[:2]
            if k == 0:
                x1a, y1a, x2a, y2a = max(xc - w, 0), max(yc - h, 0), xc, yc
                x1b, y1b, x2b, y2b = w - (x2a - x1a), h - (y2a - y1a), w, h
            elif k == 1:
                x1a, y1a, x2a, y2a = xc, max(yc - h, 0), min(xc + w, 2 * s), yc
                x1b, y1b, x2b, y2b = 0, h - (y2a - y1a), min(w, x2a - x1a), h
            elif k == 2:
                x1a, y1a, x2a, y2a = max(xc - w, 0), yc, xc, min(2 * s, yc + h)
                x1b, y1b, x2b, y2b = w - (x2a - x1a), 0, w, min(y2a - y1a, h)
            else:
                x1a, y1a, x2a, y2a = xc, yc, min(xc + w, 2 * s), min(2 * s, yc + h)
                x1b, y1b, x2b, y2b = 0, 0, min(w, x2a - x1a), min(y2a - y1a, h)
            khung[y1a:y2a, x1a:x2a] = img[y1b:y2b, x1b:x2b]
            kmask[y1a:y2a, x1a:x2a] = mask[y1b:y2b, x1b:x2b]
        return affine_ngau_nhien(khung, kmask, s, AUG["scale"], AUG["translate"])

    def __getitem__(self, i):
        if self.augment:
            if self.mosaic and random.random() < AUG["mosaic"]:
                img, mask = self.mosaic4(i)
            else:
                img, mask = affine_ngau_nhien(*self.anh_vuong(i), self.s, AUG["scale"], AUG["translate"])
            img = hsv_ngau_nhien(img, AUG["hsv_h"], AUG["hsv_s"], AUG["hsv_v"])
            if random.random() < AUG["fliplr"]:
                img, mask = np.ascontiguousarray(img[:, ::-1]), np.ascontiguousarray(mask[:, ::-1])
        else:
            img, mask = self.anh_vuong(i)

        img = img.astype(np.float32) / 255.0
        if self.chuan_hoa == "imagenet":  # PraNet dùng backbone ResNet nên chuẩn hóa ImageNet
            img = (img - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        mask_tensor = torch.tensor(mask > 127, dtype=torch.float32).unsqueeze(0)
        return img_tensor, mask_tensor


def khoi_tao_worker(worker_id):
    """Mỗi worker có seed riêng nhưng lặp lại được theo SEED."""
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws)
    random.seed(ws)


train_ds = PolypBG20(DATASET_ROOT, "train", IMG_SIZE, augment=True, chuan_hoa=CHUAN_HOA)
val_ds = PolypBG20(DATASET_ROOT, "val", IMG_SIZE, augment=False, chuan_hoa=CHUAN_HOA)
print(f"Tập train: {len(train_ds)} ảnh (880 polyp + 160 nền), có tăng cường")
print(f"Tập val  : {len(val_ds)} ảnh (120 polyp + 40 nền), không tăng cường")
assert len(train_ds) == 1040 and len(val_ds) == 160, "Sai số ảnh BG20"

# Kiểm tra nhanh: một mẫu có tăng cường đúng kích thước và mặt nạ chỉ có 0/1
_x, _m = train_ds[0]
assert _x.shape == (3, IMG_SIZE, IMG_SIZE) and _m.shape == (1, IMG_SIZE, IMG_SIZE)
assert set(torch.unique(_m).tolist()) <= {0.0, 1.0}
print("✅ Dataset và tăng cường hoạt động đúng")


In [ ]:
# ============================================================
# CELL 2: KIẾN TRÚC VÀ HÀM MẤT MÁT PRANET (GIỮ NGUYÊN NHƯ NOTEBOOK CŨ)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models

BATCH_SIZE = 8
LR = 1e-4
CKPT_NAME = "pranet_best_s{}.pth"

# 3. CÁC MODULE CƠ BẢN CỦA PRANET
class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class RFB(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1)
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), dim=1))
        return self.relu(x_cat + self.conv_res(x))
class Aggregation(nn.Module):
    def __init__(self, channel=32):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        
        # Nhận vào x2_2 sau khi upsample 2x lên mức spatial của x3 (44x44)
        self.conv_concat = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat2 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3 * channel, 1, 1)

    def forward(self, x1, x2, x3):
        # x1: 11x11, x2: 22x22, x3: 44x44
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2                      # 22x22
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) * self.conv_upsample3(self.upsample(x2)) * x3  # 44x44

        # Ghép x2_1 (22x22) và x1_1 upsampled (22x22) -> x2_2 (22x22)
        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_upsample5(x2_2)                                         # (B, 64, 22, 22)

        # Upsample x2_2 từ 22x22 lên 44x44 để khớp với x3_1 (44x44)
        x2_2_up = self.conv_concat(self.upsample(x2_2))                         # (B, 64, 44, 44)

        # Ghép x3_1 (32 channels, 44x44) + x2_2_up (64 channels, 44x44) -> 96 channels, 44x44
        x3_2 = torch.cat((x3_1, x2_2_up), 1)
        x3_2 = self.conv_concat2(x3_2)

        x = self.conv5(x3_2)                                                    # (B, 1, 44, 44)
        return x

class ReverseAttention(nn.Module):
    def __init__(self, channel=32):
        super().__init__()
        self.conv1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv3 = nn.Conv2d(channel, 1, 1)

    def forward(self, x, bg):
        # Reverse attention: (1 - Sigmoid(bg)) * x
        x_att = -1 * (torch.sigmoid(bg)) + 1
        x = x * x_att
        x = self.conv1(x)
        x = self.conv2(x)
        return self.conv3(x)

# 4. TOÀN BỘ MẠNG PRANET
class PraNet(nn.Module):
    def __init__(self, channel=32):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1  # 256
        self.layer2 = resnet.layer2  # 512
        self.layer3 = resnet.layer3  # 1024
        self.layer4 = resnet.layer4  # 2048

        self.rfb2 = RFB(512, channel)
        self.rfb3 = RFB(1024, channel)
        self.rfb4 = RFB(2048, channel)

        self.agg = Aggregation(channel)
        self.ra4 = ReverseAttention(channel)
        self.ra3 = ReverseAttention(channel)
        self.ra2 = ReverseAttention(channel)

    def forward(self, x):
        h, w = x.shape[2:]
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)  # 512
        x3 = self.layer3(x2)  # 1024
        x4 = self.layer4(x3)  # 2048

        x2_rfb = self.rfb2(x2)  # (B, 32, H/8, W/8)
        x3_rfb = self.rfb3(x3)  # (B, 32, H/16, W/16)
        x4_rfb = self.rfb4(x4)  # (B, 32, H/32, W/32)

        # Global Saliency Map từ Aggregation
        sg = self.agg(x4_rfb, x3_rfb, x2_rfb)
        lateral_map_sg = F.interpolate(sg, size=(h, w), mode='bilinear', align_corners=False)

        # Reverse Attention Stage 4
        s4 = self.ra4(x4_rfb, F.interpolate(sg, size=x4_rfb.shape[2:], mode='bilinear', align_corners=False))
        lateral_map_s4 = F.interpolate(s4, size=(h, w), mode='bilinear', align_corners=False)

        # Reverse Attention Stage 3
        s3 = self.ra3(x3_rfb, F.interpolate(s4, size=x3_rfb.shape[2:], mode='bilinear', align_corners=False))
        lateral_map_s3 = F.interpolate(s3, size=(h, w), mode='bilinear', align_corners=False)

        # Reverse Attention Stage 2
        s2 = self.ra2(x2_rfb, F.interpolate(s3, size=x2_rfb.shape[2:], mode='bilinear', align_corners=False))
        lateral_map_s2 = F.interpolate(s2, size=(h, w), mode='bilinear', align_corners=False)

        return lateral_map_s2, lateral_map_s3, lateral_map_s4, lateral_map_sg

# 5. HÀM LOSS (STRUCTURE LOSS: BCE + IOU)
def structure_loss(pred, mask):
    bce = F.binary_cross_entropy_with_logits(pred, mask, reduction='mean')
    pred_prob = torch.sigmoid(pred)
    inter = (pred_prob * mask).sum(dim=(2, 3))
    union = (pred_prob + mask).sum(dim=(2, 3))
    iou = 1.0 - (inter + 1.0) / (union - inter + 1.0)
    return bce + iou.mean()


In [ ]:
# ============================================================
# CELL 3: KHÓA TẤT ĐỊNH VÀ HUẤN LUYỆN PRANET (640, CÓ TĂNG CƯỜNG, ĐÁNH GIÁ MỖI EPOCH)
# ============================================================
import os, csv, time, random
import numpy as np
import torch
from torch.utils.data import DataLoader

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
RUN_NAME = f"{MODEL_NAME}_640_aug_s{SEED}"
OUTPUT_DIR = f"/kaggle/working/runs/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

g = torch.Generator()
g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
                          worker_init_fn=khoi_tao_worker, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = PraNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

csv_path = os.path.join(OUTPUT_DIR, "results.csv")
with open(csv_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch", "train_loss", "val_dice", "val_iou", "lr", "mosaic", "time_s"])

print(f"Bắt đầu huấn luyện {MODEL_NAME} trên {DEVICE}: SEED={SEED}, IMG_SIZE={IMG_SIZE}, EPOCHS={EPOCHS}")
best_val_dice, best_epoch, t0 = -1.0, 0, time.time()

for epoch in range(1, EPOCHS + 1):
    # Tắt mosaic ở CLOSE_MOSAIC epoch cuối (worker được tạo lại mỗi epoch nên nhận giá trị mới)
    train_ds.mosaic = AUG["mosaic"] > 0 and epoch <= EPOCHS - CLOSE_MOSAIC

    model.train()
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        out2, out3, out4, out_g = model(imgs)
        # Kết hợp loss từ cả 4 nhánh theo paper PraNet
        loss = structure_loss(out2, masks) + structure_loss(out3, masks) + \
               structure_loss(out4, masks) + structure_loss(out_g, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()
    train_loss = total_loss / len(train_ds)

    # Đánh giá trên tập val mỗi epoch (Dice, IoU theo từng ảnh; ảnh nền: đúng nếu không báo điểm nào)
    model.eval()
    dices, ious = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits, _, _, _ = model(imgs)
            probs = torch.sigmoid(logits) > 0.5
            for p, gt in zip(probs, masks > 0.5):
                inter = (p & gt).sum().item()
                union = (p | gt).sum().item()
                tong = p.sum().item() + gt.sum().item()
                if gt.sum().item() == 0:
                    d = iou = 1.0 if p.sum().item() == 0 else 0.0
                else:
                    d, iou = 2.0 * inter / (tong + 1e-5), inter / (union + 1e-5)
                dices.append(d)
                ious.append(iou)
    val_dice, val_iou = float(np.mean(dices)), float(np.mean(ious))
    elapsed = time.time() - t0

    with open(csv_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, f"{train_loss:.6f}", f"{val_dice:.6f}", f"{val_iou:.6f}",
                                f"{lr_now:.8f}", int(train_ds.mosaic), f"{elapsed:.1f}"])

    mark = ""
    if val_dice > best_val_dice:
        best_val_dice, best_epoch = val_dice, epoch
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, CKPT_NAME.format(SEED)))
        mark = " (BEST SAVED)"
    if epoch % 5 == 0 or epoch == 1 or mark:
        print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] loss {train_loss:.4f} | Val Dice {val_dice:.4f} | "
              f"Val IoU {val_iou:.4f} | {elapsed / 60:.1f} phút{mark}")

torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "last.pth"))
with open(os.path.join(OUTPUT_DIR, "tom_tat.txt"), "w") as f:
    f.write(f"model={MODEL_NAME} seed={SEED} img_size={IMG_SIZE} epochs={EPOCHS} aug={AUG}\n")
    f.write(f"best_epoch={best_epoch} best_val_dice={best_val_dice:.6f} train_time_s={time.time() - t0:.1f}\n")
print(f"\nXong. Val Dice tốt nhất {best_val_dice:.4f} ở epoch {best_epoch}; thời gian {(time.time() - t0) / 3600:.2f} giờ")


In [ ]:
# ============================================================
# CELL 4: NÉN KẾT QUẢ ĐỂ TẢI VỀ (tab Output)
# ============================================================
!cd /kaggle/working && zip -r "runs_{RUN_NAME}.zip" runs
